In [0]:
CATALOG = "dbr_dev_ua5816bd"
SCHEMA = "mialkovska_viktor594"
VOLUME = "raw_files"

RAW_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
DATASET_PATH = f"{RAW_PATH}/tristar"

staging_path = f"{DATASET_PATH}/staging"

In [0]:

import json
import requests
from datetime import datetime

url = "https://ckan2.multimediagdansk.pl/gpsPositions?v=2"

def get_tristar_snapshot():
    response = requests.get(url, timeout=20)
    response.raise_for_status()

    data = response.json()

    vehicles = data["vehicles"]

    timestamp = (
        datetime.fromisoformat(data["lastUpdate"].replace("Z", "+00:00"))
        .strftime("%Y%m%d_%H%M%S")
    )

    return vehicles, timestamp


def save_snapshot(vehicles, timestamp, transform=None):
    for vehicle in vehicles:
        record = vehicle.copy()

        if transform:
            record = transform(record)

        file_name = f"{record['vehicleId']}_{timestamp}.json"
        file_path = f"{staging_path}/{file_name}"

        with open(file_path, "w") as f:
            json.dump(record, f, indent=2, ensure_ascii=False)

def add_vehicle_type(record):
    record["vehicleType"] = (
        "bus" if record["vehicleId"] % 2 == 0 else "tram"
    )

    return record


In [0]:
vehicles, timestamp = get_tristar_snapshot()

save_snapshot(vehicles, timestamp)

In [0]:
%skip
dbutils.fs.rm(staging_path, True)
dbutils.fs.mkdirs(staging_path)

In [0]:
vehicles, timestamp = get_tristar_snapshot()

save_snapshot(
    vehicles,
    timestamp,
    transform=add_vehicle_type
)

print(f"Files with new schema created: {len(vehicles)}")

In [0]:
vehicles, timestamp = get_tristar_snapshot()

error_record_1 = vehicles[0].copy()
error_record_2 = vehicles[1].copy()

error_record_1["speed"] = "unknown"
error_record_2["lat"] = "unknown"

error_records = [error_record_1, error_record_2]

for i, record in enumerate(error_records, start=1):
    file_name = f"error_{i}_{record['vehicleId']}_{timestamp}.json"
    file_path = f"{staging_path}/{file_name}"

    with open(file_path, "w") as f:
        json.dump(record, f, indent=2, ensure_ascii=False)

print(f"Error files created: {len(error_records)}")